# Liu2024 TWFB + DGFMDRM — pyRiemann Reproduction

**Goal:** Reproduce (or closely approximate) the Liu2024 TWFB+DGFMDRM method using Python and pyRiemann.

**What Liu2024 actually did (from the paper + MATLAB source):**
- The MATLAB file uses freq_bands = {[8,12],[8,20],[8,30],[12,20],[15,20],[15,30],[20,30],[8,15]} and
  picks the best band per 10-fold random split — **not** the 19-band TWFB in Table 3 of the paper.
  The paper's Table 3 describes the full algorithm; the provided MATLAB is a simplified 8-band version.
- Classification is via `fgmdm` = Fisher Geodesic MDM (discriminant geodesic filtering + MDM).
- Time window: 2s of data starting 800 samples before trigger at 500 Hz → 800:2800 → 2000 samples.
  At 500 Hz that is samples 0–4s of the MI period (the trigger marks MI onset at 800 samples in).

**Our Python approximation:**
- MODE `broad_8_30_mdm`: Single 8–30 Hz band, covariances + MDM. Fast sanity check.
- MODE `filterbank_tangent_lda`: All filter-band × time-window combos, tangent-space concat + shrinkage LDA.
- MODE `twfb_inner_selection`: Inner-fold validation selects best band/window combo (leakage-safe).
- MODE `matlab_faithful_attempt`: Matches the MATLAB 8-band set + MDM, random split repeated 10×.

**Deviations from MATLAB:**
1. MATLAB uses raw covariance X'X (not normalised). We use OAS or empirical estimator.
2. MATLAB `fgmdm` = Fisher Geodesic MDM — we approximate with TangentSpace + LDA or plain MDM.
3. MATLAB re-randomises splits every repeat. We use StratifiedShuffleSplit for reproducibility.
4. Paper's 19-band TWFB is in `filterbank_tangent_lda`; MATLAB's 8-band is in `matlab_faithful_attempt`.

## 1. Imports

In [ ]:
import os
import re
import sys
import json
import random
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy import signal as sp_signal

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from pyriemann.utils.mean import mean_covariance

warnings.filterwarnings('ignore', category=RuntimeWarning)
print(f"numpy {np.__version__}, pandas {pd.__version__}")
try:
    import pyriemann
    print(f"pyriemann {pyriemann.__version__}")
except Exception:
    print("pyriemann not found — install with: pip install pyriemann")

## 2. CONFIG

In [ ]:
CONFIG = {
    # --- Paths ---
    "data_root": "../../liu2024_data/liu2024_figshare/sourcedata",  # adjust to your path
    "artifact_root": "../../artifacts/liu2024_twfb_dgfmrdm_pyriemann",

    # --- Dataset ---
    "subjects": "all",           # "all" or list of ints e.g. [1, 2, 3]
    "random_state": 2026,
    "sfreq_raw": 500,

    # --- MI window (seconds, relative to MI onset at t=0) ---
    "mi_window_s": (0.0, 4.0),

    # --- Cross-validation ---
    "n_repeats": 10,
    "test_size": 0.40,           # 40 trials * 0.40 = 16 test, 24 train (matches Liu2024)

    # --- Covariance estimation ---
    "cov_estimator": "oas",      # 'oas', 'lwf', 'scm' (scm = raw X'X like MATLAB)
    "cov_reg_eps": 1e-6,         # added to diagonal to ensure SPD

    # --- Mode ---
    # Options:
    #   'broad_8_30_mdm'         : single band, MDM (fastest)
    #   'filterbank_tangent_lda' : paper's 19 bands x 7 windows, tangent + shrinkage LDA
    #   'twfb_inner_selection'   : inner-fold selects best band+window (leakage-safe)
    #   'matlab_faithful_attempt': 8 MATLAB bands, MDM, matches MATLAB structure
    "mode": "filterbank_tangent_lda",

    # --- Filter banks (paper Table 3 — 19 bands) ---
    "filter_bands_hz": [
        (8, 12), (9, 13), (10, 14), (11, 15), (12, 16),
        (13, 17), (14, 18), (15, 19), (16, 20), (17, 21),
        (18, 22), (19, 23), (20, 24), (21, 25), (22, 26),
        (23, 27), (24, 28), (25, 29), (26, 30),
    ],

    # --- MATLAB 8-band set (for matlab_faithful_attempt mode) ---
    "matlab_bands_hz": [
        (8, 12), (8, 20), (8, 30), (12, 20),
        (15, 20), (15, 30), (20, 30), (8, 15),
    ],

    # --- Time windows (seconds, relative to MI onset) ---
    "time_windows_s": [
        (0.0, 1.0), (0.5, 1.5), (1.0, 2.0), (1.5, 2.5),
        (2.0, 3.0), (2.5, 3.5), (3.0, 4.0),
    ],

    # --- Inner CV for twfb_inner_selection mode ---
    "use_inner_selection": True,
    "inner_cv_splits": 3,

    # --- Classifier ---
    "classifier": "tangent_shrinkage_lda",  # or 'mdm'
}

DATA_ROOT = Path(CONFIG["data_root"])
ARTIFACT_ROOT = Path(CONFIG["artifact_root"])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

SFREQ = int(CONFIG["sfreq_raw"])   # We work at raw 500 Hz for Riemannian (no resampling needed)
MI_START = int(CONFIG["mi_window_s"][0] * SFREQ)
MI_STOP  = int(CONFIG["mi_window_s"][1] * SFREQ)
MI_SAMPLES = MI_STOP - MI_START

np.random.seed(CONFIG["random_state"])
random.seed(CONFIG["random_state"])

print(f"Mode:         {CONFIG['mode']}")
print(f"Data root:    {DATA_ROOT}")
print(f"Artifacts:    {ARTIFACT_ROOT}")
print(f"SFREQ:        {SFREQ} Hz")
print(f"MI window:    {CONFIG['mi_window_s']} → {MI_SAMPLES} samples")
print(f"n_repeats:    {CONFIG['n_repeats']}, test_size: {CONFIG['test_size']}")

## 3. Liu2024 Channel Constants

In [ ]:
# Liu2024 source MAT: trials x 33 channels x 4000 samples at 500 Hz
# Channel 17 (0-indexed) = CPz source reference → drop
# Channels 30, 31 = EOG → drop
# Channel 32 = marker → drop
# Keep 29 EEG channels

SOURCE_EEG_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",          # idx 17 = CPz ref
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CPZ_IDX = 17
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]   # 29 channels
EEG_NAMES = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS = len(EEG_KEEP_IDX)  # 29

# NOTE: The MATLAB file uses channel = [1:17 19:30] (1-indexed) which is indices 0:17, 18:30
# in 0-indexed Python = same 29 channels (drops index 17 = CPz). Consistent.

print(f"EEG channels: {N_CHANS}")
print(f"Channel names: {EEG_NAMES}")

## 4. Data Loading

In [ ]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    return int(nums[-1]) if nums else None


def load_subject(mat_path):
    """Load one Liu2024 subject. Returns X (40, 29, MI_SAMPLES) and y (40,) zero-based."""
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)

    # Try common key names
    raw = None
    for key in ["rawdata", "data", "EEG", "eeg"]:
        if key in mat:
            raw = mat[key]
            break
    if raw is None:
        # Walk nested struct
        for k, v in mat.items():
            if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim == 3:
                raw = v
                break
    if raw is None:
        raise ValueError(f"Cannot find rawdata in {mat_path}")

    raw = np.asarray(raw, dtype=np.float64)
    # Ensure shape trials x channels x samples
    if raw.shape[0] == 33 or raw.shape[0] == 40:
        # might be channels x samples x trials or similar
        pass
    # Find trial axis (should be 40)
    trial_ax = None
    for ax, sz in enumerate(raw.shape):
        if sz == 40:
            trial_ax = ax
            break
    if trial_ax is None:
        raise ValueError(f"Cannot find trial axis (40) in shape {raw.shape} from {mat_path}")
    raw = np.moveaxis(raw, trial_ax, 0)  # → 40 x ? x ?
    # Find channel axis (33)
    if raw.shape[1] == 33:
        pass  # 40 x 33 x 4000 — correct
    elif raw.shape[2] == 33:
        raw = raw.transpose(0, 2, 1)  # 40 x 33 x 4000
    else:
        raise ValueError(f"Cannot find 33-channel axis in {raw.shape}")

    assert raw.shape == (40, 33, 4000), f"Unexpected shape after normalisation: {raw.shape}"

    # Select 29 EEG channels, then MI window
    X = raw[:, EEG_KEEP_IDX, :]            # 40 x 29 x 4000
    X = X[:, :, MI_START:MI_STOP]          # 40 x 29 x MI_SAMPLES

    # Labels
    labels = mat.get("labels", mat.get("label", None))
    if labels is None:
        raise ValueError(f"Cannot find labels in {mat_path}")
    y = np.asarray(labels, dtype=int).ravel()
    # Map 1/2 → 0/1
    if set(np.unique(y).tolist()).issubset({1, 2}):
        y = y - 1

    assert X.shape == (40, N_CHANS, MI_SAMPLES), f"Bad X shape: {X.shape}"
    assert len(y) == 40
    assert np.isfinite(X).all(), "Non-finite values in EEG data"

    return X.astype(np.float64), y


def find_mat_files(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Data root not found: {root}")
    files = sorted(root.rglob("*.mat"))
    if not files:
        raise FileNotFoundError(f"No .mat files under {root}")
    return files


mat_files = find_mat_files(DATA_ROOT)
all_sids = sorted({subject_id_from_path(f) for f in mat_files})

if CONFIG["subjects"] == "all":
    SUBJECT_IDS = all_sids
else:
    SUBJECT_IDS = sorted(int(s) for s in CONFIG["subjects"])

# Build a lookup: sid → mat path
sid_to_path = {}
for f in mat_files:
    sid = subject_id_from_path(f)
    if sid in SUBJECT_IDS:
        sid_to_path[sid] = f

print(f"Found {len(mat_files)} .mat files, using {len(SUBJECT_IDS)} subjects")
print(f"Subject IDs: {SUBJECT_IDS}")

## 5. Signal Processing Utilities

**Leakage note:** Bandpass filtering is applied per-trial to the raw signal. This is safe — the filter is a fixed transform (no data-driven parameters). Covariance means are computed on training-fold trials only.

In [ ]:
def bandpass_trial(X_trial, low, high, sfreq, order=4):
    """Zero-phase Butterworth bandpass for one trial (n_chans x n_times)."""
    nyq = sfreq / 2.0
    b, a = sp_signal.butter(order, [low / nyq, high / nyq], btype="band")
    return sp_signal.filtfilt(b, a, X_trial, axis=-1)


def bandpass_all(X, low, high, sfreq=500, order=4):
    """Bandpass all trials. X: (n_trials, n_chans, n_times)."""
    out = np.empty_like(X)
    for i in range(len(X)):
        out[i] = bandpass_trial(X[i], low, high, sfreq, order)
    return out


def window_trials(X, t_start, t_stop, sfreq=500):
    """Crop trials to a time window (in seconds relative to MI_START)."""
    s0 = int(t_start * sfreq)
    s1 = int(t_stop * sfreq)
    return X[:, :, s0:s1]


def compute_covs(X, estimator="oas", reg_eps=1e-6):
    """
    Compute SPD covariance matrices.
    X: (n_trials, n_chans, n_times)
    Returns: (n_trials, n_chans, n_chans)
    """
    cov_obj = Covariances(estimator=estimator)
    C = cov_obj.fit_transform(X.astype(np.float64))
    # Regularise: add eps * I to guarantee SPD
    if reg_eps > 0:
        I = np.eye(C.shape[-1])
        C = C + reg_eps * I[np.newaxis]
    assert np.all(np.isfinite(C)), "Non-finite covariance matrices"
    return C


# Quick shape smoke test
_x = np.random.randn(4, 29, 500).astype(np.float64)
_xf = bandpass_all(_x, 8, 30, sfreq=500)
_c = compute_covs(_xf, "oas")
assert _c.shape == (4, 29, 29), f"Cov shape wrong: {_c.shape}"
print("Signal processing utilities OK — cov shape:", _c.shape)

## 6. Classifier Pipelines

In [ ]:
def make_clf(mode="tangent_shrinkage_lda", metric="riemann"):
    """Return a fitted-on-covs classifier."""
    if mode == "mdm":
        return MDM(metric=metric)
    elif mode == "tangent_shrinkage_lda":
        # TangentSpace projects SPD covs to Euclidean tangent vectors,
        # then shrinkage LDA classifies. This approximates DGFMDRM.
        return Pipeline([
            ("ts", TangentSpace(metric=metric)),
            ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
        ])
    else:
        raise ValueError(f"Unknown classifier mode: {mode}")


def fit_predict_covs(C_train, y_train, C_test, clf_mode, metric="riemann"):
    """Fit classifier on training covs and predict test covs."""
    assert len(np.unique(y_train)) == 2, "Expected binary labels"
    clf = make_clf(clf_mode, metric)
    clf.fit(C_train, y_train)
    return clf.predict(C_test)


print("Classifier pipelines defined.")

## 7. Feature Extraction Strategies per Mode

In [ ]:
def extract_features_broad(X_train, X_test, cfg):
    """
    Broad 8-30 Hz band, full MI window.
    Returns C_train (n_train, C, C) and C_test (n_test, C, C).
    Fit only on train.
    """
    low, high = 8, 30
    Xf_train = bandpass_all(X_train, low, high, SFREQ)
    Xf_test  = bandpass_all(X_test,  low, high, SFREQ)
    C_train = compute_covs(Xf_train, cfg["cov_estimator"], cfg["cov_reg_eps"])
    C_test  = compute_covs(Xf_test,  cfg["cov_estimator"], cfg["cov_reg_eps"])
    return C_train, C_test


def extract_features_filterbank_concat(X_train, X_test, cfg, bands, windows):
    """
    All bands × windows. Project each to tangent space (fitted on train),
    then concatenate all tangent vectors.
    Fit only on train covariances.
    Returns feat_train (n_train, D), feat_test (n_test, D).
    """
    feats_train, feats_test = [], []
    for (low, high) in bands:
        Xf_train = bandpass_all(X_train, low, high, SFREQ)
        Xf_test  = bandpass_all(X_test,  low, high, SFREQ)
        for (t0, t1) in windows:
            Xw_train = window_trials(Xf_train, t0, t1, SFREQ)
            Xw_test  = window_trials(Xf_test,  t0, t1, SFREQ)
            C_train = compute_covs(Xw_train, cfg["cov_estimator"], cfg["cov_reg_eps"])
            C_test  = compute_covs(Xw_test,  cfg["cov_estimator"], cfg["cov_reg_eps"])
            # Fit tangent space on training covs only
            ts = TangentSpace(metric="riemann")
            f_train = ts.fit_transform(C_train)
            f_test  = ts.transform(C_test)
            feats_train.append(f_train)
            feats_test.append(f_test)

    feat_train = np.concatenate(feats_train, axis=1)
    feat_test  = np.concatenate(feats_test,  axis=1)
    return feat_train, feat_test


def select_best_combo_inner(X_train, y_train, cfg, bands, windows, n_inner=3):
    """
    Inner-fold selection of best (band, window) pair.
    Uses only X_train/y_train — no test leakage.
    Returns (best_band, best_window, best_inner_acc).
    """
    inner_cv = StratifiedKFold(n_splits=n_inner, shuffle=True,
                               random_state=cfg["random_state"])
    best_acc, best_combo = -1, None
    for (low, high) in bands:
        Xf = bandpass_all(X_train, low, high, SFREQ)
        for (t0, t1) in windows:
            Xw = window_trials(Xf, t0, t1, SFREQ)
            fold_accs = []
            for tr_idx, va_idx in inner_cv.split(Xw, y_train):
                C_tr = compute_covs(Xw[tr_idx], cfg["cov_estimator"], cfg["cov_reg_eps"])
                C_va = compute_covs(Xw[va_idx], cfg["cov_estimator"], cfg["cov_reg_eps"])
                try:
                    preds = fit_predict_covs(C_tr, y_train[tr_idx], C_va,
                                            cfg["classifier"])
                    fold_accs.append(balanced_accuracy_score(y_train[va_idx], preds))
                except Exception:
                    fold_accs.append(0.5)
            mean_acc = float(np.mean(fold_accs))
            if mean_acc > best_acc:
                best_acc = mean_acc
                best_combo = ((low, high), (t0, t1))
    return best_combo[0], best_combo[1], best_acc


print("Feature extraction strategies defined.")

## 8. Per-Subject Cross-Validation Runner

**Leakage controls enforced here:**
- Bandpass filter is a fixed transform (no fitting on data) → safe to apply before splitting.
- Covariance matrices are computed per split, per fold.
- TangentSpace reference point is fitted on training covs only, then applied to test covs.
- Inner band/window selection (when enabled) uses only training data.
- Classifier is fitted on training data only.

In [ ]:
def collapse_diagnostics(y_pred, n_classes=2):
    counts = np.bincount(y_pred, minlength=n_classes)
    dominant = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {
        "collapse_flag": bool(dominant > 0.95),
        "collapse_ratio": float(dominant),
        "pred_counts": counts.tolist(),
    }


def run_subject(sid, X, y, cfg):
    """Run repeated stratified splits for one subject."""
    mode = cfg["mode"]
    bands = cfg["filter_bands_hz"] if mode != "matlab_faithful_attempt" else cfg["matlab_bands_hz"]
    windows = cfg["time_windows_s"]

    sss = StratifiedShuffleSplit(
        n_splits=cfg["n_repeats"],
        test_size=cfg["test_size"],
        random_state=cfg["random_state"],
    )

    fold_results = []
    for fold_idx, (tr_idx, te_idx) in enumerate(sss.split(X, y)):
        X_train, X_test = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]

        assert len(np.unique(y_train)) == 2, f"Sub {sid} fold {fold_idx}: only one class in train"

        selected_band, selected_window = None, None

        try:
            if mode == "broad_8_30_mdm":
                C_train, C_test = extract_features_broad(X_train, X_test, cfg)
                y_pred = fit_predict_covs(C_train, y_train, C_test, "mdm")

            elif mode == "filterbank_tangent_lda":
                feat_train, feat_test = extract_features_filterbank_concat(
                    X_train, X_test, cfg, bands, windows)
                lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
                lda.fit(feat_train, y_train)
                y_pred = lda.predict(feat_test)

            elif mode == "twfb_inner_selection":
                best_band, best_window, _ = select_best_combo_inner(
                    X_train, y_train, cfg, bands, windows,
                    n_inner=cfg["inner_cv_splits"])
                selected_band, selected_window = best_band, best_window
                Xf_train = bandpass_all(X_train, *best_band, SFREQ)
                Xf_test  = bandpass_all(X_test,  *best_band, SFREQ)
                Xw_train = window_trials(Xf_train, *best_window, SFREQ)
                Xw_test  = window_trials(Xf_test,  *best_window, SFREQ)
                C_train = compute_covs(Xw_train, cfg["cov_estimator"], cfg["cov_reg_eps"])
                C_test  = compute_covs(Xw_test,  cfg["cov_estimator"], cfg["cov_reg_eps"])
                y_pred = fit_predict_covs(C_train, y_train, C_test, cfg["classifier"])

            elif mode == "matlab_faithful_attempt":
                # Mimic MATLAB: try all 8 bands, pick best on this split's test set.
                # WARNING: this mimics the MATLAB code which selects on the test fold.
                # This is what Liu2024 did. We flag it clearly.
                best_acc_inner, best_pred = -1, None
                for (low, high) in bands:
                    Xf_train = bandpass_all(X_train, low, high, SFREQ)
                    Xf_test  = bandpass_all(X_test,  low, high, SFREQ)
                    # Full MI window at 500 Hz (0–4s)
                    C_train = compute_covs(Xf_train, "scm", cfg["cov_reg_eps"])
                    C_test  = compute_covs(Xf_test,  "scm", cfg["cov_reg_eps"])
                    try:
                        preds = fit_predict_covs(C_train, y_train, C_test, "mdm")
                        acc = accuracy_score(y_test, preds)
                        if acc > best_acc_inner:
                            best_acc_inner = acc
                            best_pred = preds
                            selected_band = (low, high)
                    except Exception:
                        pass
                y_pred = best_pred if best_pred is not None else np.zeros(len(y_test), dtype=int)

            else:
                raise ValueError(f"Unknown mode: {mode}")

        except Exception as exc:
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            y_pred = np.zeros(len(y_test), dtype=int)

        acc = float(accuracy_score(y_test, y_pred))
        bacc = float(balanced_accuracy_score(y_test, y_pred))
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
        diag = collapse_diagnostics(y_pred)

        # Per-class recall
        cm_arr = np.array(cm)
        left_recall  = cm_arr[0, 0] / cm_arr[0].sum() if cm_arr[0].sum() > 0 else float("nan")
        right_recall = cm_arr[1, 1] / cm_arr[1].sum() if cm_arr[1].sum() > 0 else float("nan")

        fold_results.append({
            "subject_id":       sid,
            "fold_id":          fold_idx,
            "accuracy":         acc,
            "balanced_accuracy": bacc,
            "left_recall":      float(left_recall),
            "right_recall":     float(right_recall),
            "confusion_matrix": cm,
            "collapse_flag":    diag["collapse_flag"],
            "collapse_ratio":   diag["collapse_ratio"],
            "pred_counts":      diag["pred_counts"],
            "n_train":          int(len(y_train)),
            "n_test":           int(len(y_test)),
            "selected_band":    str(selected_band),
            "selected_window":  str(selected_window),
        })

    return fold_results


print("Subject runner defined.")

## 9. Run All Subjects

In [ ]:
ALL_FOLD_RESULTS = []
SUBJECT_SUMMARIES = []

print(f"Running {len(SUBJECT_IDS)} subjects | mode={CONFIG['mode']}")
print("=" * 60)

for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file found, skipping")
        continue

    try:
        X, y = load_subject(mat_path)
    except Exception as exc:
        print(f"  Sub {sid:02d}: load error — {exc}")
        continue

    fold_res = run_subject(sid, X, y, CONFIG)
    ALL_FOLD_RESULTS.extend(fold_res)

    accs  = [r["accuracy"] for r in fold_res]
    baccs = [r["balanced_accuracy"] for r in fold_res]
    collapses = sum(1 for r in fold_res if r["collapse_flag"])
    mean_bacc = np.mean(baccs)

    SUBJECT_SUMMARIES.append({
        "subject_id":             sid,
        "mean_accuracy":          float(np.mean(accs)),
        "std_accuracy":           float(np.std(accs)),
        "mean_balanced_accuracy": float(mean_bacc),
        "std_balanced_accuracy":  float(np.std(baccs)),
        "n_folds":                len(fold_res),
        "n_collapsed_folds":      collapses,
    })

    print(f"  Sub {sid:02d}: bal_acc={mean_bacc*100:.1f}% ± {np.std(baccs)*100:.1f}%  "
          f"collapse={collapses}/{len(fold_res)}")

print("=" * 60)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

## 10. Aggregate Results

In [ ]:
fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)

all_baccs = fold_df["balanced_accuracy"].values
all_accs  = fold_df["accuracy"].values
n_collapsed = fold_df["collapse_flag"].sum()

# Aggregate confusion matrix
cm_total = np.zeros((2, 2), dtype=int)
for row in ALL_FOLD_RESULTS:
    cm_total += np.array(row["confusion_matrix"])

global_summary = {
    "mode":                   CONFIG["mode"],
    "n_subjects":             len(SUBJECT_SUMMARIES),
    "n_folds_total":          len(ALL_FOLD_RESULTS),
    "mean_accuracy":          float(np.mean(all_accs)),
    "std_accuracy":           float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_baccs)),
    "std_balanced_accuracy":  float(np.std(all_baccs)),
    "n_collapsed_folds":      int(n_collapsed),
    "collapse_rate":          float(n_collapsed / len(ALL_FOLD_RESULTS)),
    "confusion_matrix":       cm_total.tolist(),
}

print("=" * 60)
print(f"GLOBAL RESULTS — mode={CONFIG['mode']}")
print(f"  Mean Balanced Accuracy: {global_summary['mean_balanced_accuracy']*100:.2f}% ± {global_summary['std_balanced_accuracy']*100:.2f}%")
print(f"  Mean Accuracy:          {global_summary['mean_accuracy']*100:.2f}% ± {global_summary['std_accuracy']*100:.2f}%")
print(f"  Collapsed folds:        {n_collapsed}/{len(ALL_FOLD_RESULTS)} ({global_summary['collapse_rate']*100:.1f}%)")
print(f"  Aggregated CM:")
print(f"    (rows=true, cols=pred, 0=Left, 1=Right)")
print(f"    {cm_total}")
# Per-class recall from aggregate CM
left_recall_agg  = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() > 0 else float("nan")
right_recall_agg = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() > 0 else float("nan")
print(f"  Left recall:  {left_recall_agg*100:.1f}%")
print(f"  Right recall: {right_recall_agg*100:.1f}%")
print("=" * 60)

## 11. Save Artifacts

In [ ]:
fold_df.to_csv(ARTIFACT_ROOT / "fold_results.csv", index=False)
subject_df.to_csv(ARTIFACT_ROOT / "subject_summary.csv", index=False)

with open(ARTIFACT_ROOT / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)

with open(ARTIFACT_ROOT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print(f"Saved fold_results.csv ({len(fold_df)} rows)")
print(f"Saved subject_summary.csv ({len(subject_df)} rows)")
print(f"Saved global_summary.json")

## 12. Plots

In [ ]:
# --- Confusion matrix ---
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues", aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred Left", "Pred Right"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True Left", "True Right"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center", fontsize=12)
ax.set_title(f"Aggregated CM — mode={CONFIG['mode']}")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

# --- Per-subject balanced accuracy ---
fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids  = subject_df["subject_id"].values
baccs = subject_df["mean_balanced_accuracy"].values * 100
stds  = subject_df["std_balanced_accuracy"].values * 100
ax.bar(sids, baccs, yerr=stds, capsize=3, color="steelblue", alpha=0.8)
ax.axhline(50, color="red", linestyle="--", label="Chance")
ax.axhline(float(np.mean(baccs)), color="orange", linestyle="-", label=f"Mean={np.mean(baccs):.1f}%")
ax.set_xlabel("Subject ID")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title(f"Per-subject balanced accuracy — mode={CONFIG['mode']}")
ax.legend()
ax.set_xticks(sids)
ax.set_xticklabels(sids, rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "subject_accuracy_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved subject_accuracy_plot.png")